# NB01 — Data Audit and Exploratory Morphometric Analysis

Audits data quality and produces publication-ready descriptive tables and exploratory figures. The raw file is never modified.

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, random, warnings, hashlib, platform, sys
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

ROOT = Path(r"/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1")
DATA = ROOT / "01_DATA"
NOTEBOOKS = ROOT / "02_NOTEBOOKS"
RESULTS = ROOT / "03_RESULTS"
MODELS = ROOT / "04_MODELS"
FIGURES = ROOT / "05_FIGURES"
TABLES = ROOT / "06_TABLES"
LOGS = ROOT / "07_LOGS"
EXPORTS = ROOT / "08_EXPORTS"
FINAL_ZIP = ROOT / "09_FINAL_ZIP"

for p in [DATA, NOTEBOOKS, RESULTS, MODELS, FIGURES, TABLES, LOGS, EXPORTS, FINAL_ZIP]:
    p.mkdir(parents=True, exist_ok=True)

DATASET = DATA / "INIAP_Dataset.xlsx"
SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
INNER_FOLDS = 3
TARGET = "Class"

assert DATASET.exists(), f"Dataset not found: {DATASET}"
print("ROOT:", ROOT)
print("DATASET:", DATASET)


In [ ]:

import matplotlib.pyplot as plt
from scipy import stats

OUT = RESULTS/"NB01_AUDIT_EDA"; OUT.mkdir(exist_ok=True)
FIG = FIGURES/"NB01_AUDIT_EDA"; FIG.mkdir(exist_ok=True)
TAB = TABLES/"NB01_AUDIT_EDA"; TAB.mkdir(exist_ok=True)

df_raw = pd.read_excel(DATASET)
df = df_raw.rename(columns={"AspectRation":"AspectRatio"}).copy()

features = [c for c in df.columns if c != TARGET]
assert df.shape[0] == 4000, f"Unexpected row count: {df.shape[0]}"
assert df[TARGET].nunique() == 4, f"Unexpected class count: {df[TARGET].nunique()}"

audit = {
    "rows": int(len(df)),
    "predictors": int(len(features)),
    "classes": int(df[TARGET].nunique()),
    "missing_total": int(df.isna().sum().sum()),
    "exact_duplicate_rows": int(df.duplicated().sum()),
    "class_counts": df[TARGET].value_counts().sort_index().to_dict()
}
with open(OUT/"data_audit.json","w") as f: json.dump(audit,f,indent=2)
print(audit)


In [ ]:

# Tables
missing = df.isna().sum().rename("missing").to_frame()
missing["percent"] = 100*missing["missing"]/len(df)
missing.to_csv(TAB/"missing_values.csv")

desc = df[features].describe().T
desc["cv_percent"] = 100*desc["std"]/desc["mean"].replace(0,np.nan)
desc.to_csv(TAB/"descriptive_statistics.csv")

class_dist = df[TARGET].value_counts().sort_index().rename_axis("Class").reset_index(name="n")
class_dist["percent"] = 100*class_dist["n"]/len(df)
class_dist.to_csv(TAB/"class_distribution.csv",index=False)
display(class_dist)


In [ ]:

# Correlation and mathematical redundancy checks
corr = df[features].corr(method="pearson")
corr.to_csv(TAB/"pearson_correlation_matrix.csv")

pairs=[]
for i,a in enumerate(features):
    for b in features[i+1:]:
        pairs.append((a,b,float(corr.loc[a,b]),abs(float(corr.loc[a,b]))))
pairs_df = pd.DataFrame(pairs,columns=["feature_1","feature_2","pearson_r","abs_r"]).sort_values("abs_r",ascending=False)
pairs_df.to_csv(TAB/"feature_correlation_pairs.csv",index=False)

checks = pd.DataFrame({
    "relation":[
        "AspectRatio = MajorAxisLength / MinorAxisLength",
        "Solidity = Area / ConvexArea",
        "roundness = 4*pi*Area / Perimeter^2",
        "Compactness = EquivDiameter / MajorAxisLength"
    ],
    "max_abs_error":[
        np.max(np.abs(df["AspectRatio"] - df["MajorAxisLength"]/df["MinorAxisLength"])),
        np.max(np.abs(df["Solidity"] - df["Area"]/df["ConvexArea"])),
        np.max(np.abs(df["roundness"] - 4*np.pi*df["Area"]/(df["Perimeter"]**2))),
        np.max(np.abs(df["Compactness"] - df["EquivDiameter"]/df["MajorAxisLength"]))
    ]
})
checks.to_csv(TAB/"derived_feature_identity_checks.csv",index=False)
display(pairs_df.head(15))
display(checks)


In [ ]:

# Class distribution figure
plt.figure(figsize=(7,4.5))
class_dist.set_index("Class")["n"].plot(kind="bar")
plt.ylabel("Number of grains")
plt.xlabel("Cultivar")
plt.title("Class distribution")
plt.tight_layout()
plt.savefig(FIG/"class_distribution.png",dpi=300,bbox_inches="tight")
plt.show()

# Correlation heatmap using matplotlib only
fig, ax = plt.subplots(figsize=(9,8))
im=ax.imshow(corr.values,aspect="auto")
ax.set_xticks(range(len(features))); ax.set_xticklabels(features,rotation=90)
ax.set_yticks(range(len(features))); ax.set_yticklabels(features)
fig.colorbar(im,ax=ax,label="Pearson r")
ax.set_title("Morphometric feature correlation matrix")
plt.tight_layout()
plt.savefig(FIG/"correlation_matrix.png",dpi=300,bbox_inches="tight")
plt.show()


In [ ]:

# Per-feature distributions by class
for col in features:
    plt.figure(figsize=(7,4.5))
    for cls in sorted(df[TARGET].unique()):
        vals=df.loc[df[TARGET]==cls,col].dropna().values
        plt.hist(vals,bins=30,alpha=0.35,label=str(cls),density=True)
    plt.xlabel(col); plt.ylabel("Density"); plt.title(f"{col} by cultivar")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG/f"distribution_{col}.png",dpi=220,bbox_inches="tight")
    plt.close()

df.to_csv(OUT/"analysis_dataset_standardized_names.csv",index=False)
print("NB01 outputs saved:", OUT)
